# Ticket Triage Demo: CSV in, CSV out

Reads a CSV of customer-support tickets and adds a **category**, a **priority** and a **pass-to-a-human flag** to every row.
It embeds each ticket (Voyage `voyage-4-lite`, one paid API call per batch, no language-model calls) and applies two small trained classifiers.

**How to use this notebook**
1. Run the cells from top to bottom. Cells 1-4 are free: they only check files and install packages.
2. Cell 5 is the **only paid step**. It does nothing until you set `CONFIRM_PAID_RUN = True` yourself.
3. The key `VOYAGE_API_KEY` is read from Colab *Secrets* (the key icon in the left bar). Never paste it into a cell.

**Known limits**: trained on English customer-support tweets of five companies, 8 fixed categories, short texts. Accuracy on your own data is **not known** until it is measured on tickets labelled by people. A ticket goes to a human when the category confidence is below 0.70. See `README_triage.md` in the bundle.

## 1. Upload the bundle (free)
Upload `triage_bundle.zip` when asked.

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs("/content/triage", exist_ok=True)
os.chdir("/content/triage")
up = files.upload()                      # choose triage_bundle.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall(".")
print(sorted(os.listdir(".")))

## 2. Check the files (free)
The sums below are the ones the 162 offline tests were run with. Anything else stops here.

In [ ]:
import hashlib

EXPECTED = {
    "triage.py": "0bac348863e2386be58313dcc7ead44f",
    "test_triage_policy.py": "9ca6f40c848b80ab3d4d771e1fed8846",
    "category_classifier_embeddings.joblib": "4df43a68573f3a7a1a9dfdfe27c51e4c",
    "priority_classifier_embeddings.joblib": "301b14c120d888c92f62e904ef39265d",
    "requirements-triage.txt": "401bc1a7f5515d82df1eb2e3c9f6b4e0",
    "sample_tickets.csv": "2b18344ed03900c54d772c3ea6d8d7e3",
}
bad = []
for f, want in EXPECTED.items():
    got = hashlib.md5(open(f, "rb").read()).hexdigest() if os.path.exists(f) else "MISSING"
    print(("OK      " if got == want else "DIFFERENT"), f)
    if got != want:
        bad.append(f)
assert not bad, f"Wrong or missing files: {bad}. Upload the original triage_bundle.zip."
print("All files match.")

## 3. Install the pinned packages (free)
`scikit-learn==1.9.1` is pinned because the two model files are pickles saved with that version. Run this cell **before** importing pandas or scikit-learn in this notebook.

In [ ]:
!pip install -q -r requirements-triage.txt
!pip list 2>/dev/null | grep -i -E "scikit-learn|numpy|pandas|scipy|joblib|voyageai"

## 4. Dry run (free, nothing is sent)
Checks the file, counts the rows, estimates the size. No key needed.

In [ ]:
!python triage.py sample_tickets.csv --dry-run

## 5. Real run (PAID: tiny for the sample, a fraction of a cent)

Before you run it:
- add the secret `VOYAGE_API_KEY` in Colab (left bar, key icon) and switch notebook access on for it;
- set `CONFIRM_PAID_RUN = True` below.

If the run stops, run the cell again: finished rows are not sent (or paid for) twice. If the result file already exists, the script refuses to replace it unless you add `--overwrite` (that sends and pays again).

In [ ]:
CONFIRM_PAID_RUN = False      # <- change to True to allow the paid call
INPUT_FILE = "sample_tickets.csv"

if not CONFIRM_PAID_RUN:
    print("Paid run NOT started. Set CONFIRM_PAID_RUN = True to allow it.")
else:
    from google.colab import userdata
    try:
        os.environ["VOYAGE_API_KEY"] = userdata.get("VOYAGE_API_KEY")
    except Exception as e:
        raise SystemExit("Secret VOYAGE_API_KEY not found or access not granted: add it in the Colab Secrets panel "
                         "(key icon) and switch 'Notebook access' on.")
    !python triage.py "$INPUT_FILE"

## 6. Look at the result (free)

In [ ]:
import pandas as pd

RESULT_FILE = INPUT_FILE.rsplit(".", 1)[0] + "_triaged.csv"
if not os.path.exists(RESULT_FILE):
    print("No result file yet: run the paid cell above first.")
else:
    res = pd.read_csv(RESULT_FILE, dtype=str, keep_default_na=False)
    print(len(res), "rows")
    print("\nneeds_human:", res["needs_human"].value_counts().to_dict())
    print("\nreasons:", res.loc[res["needs_human"] == "yes", "needs_human_reason"].value_counts().to_dict())
    print("\ncategories:\n", res["category"].value_counts().to_string())
    show = ["id", "text", "category", "category_confidence", "priority", "needs_human", "needs_human_reason"]
    show = [c for c in show if c in res.columns]
    pd.set_option("display.max_colwidth", 70)
    display(res[show])

## 7. Download the result (free)

In [ ]:
if os.path.exists(RESULT_FILE):
    files.download(RESULT_FILE)
else:
    print("No result file yet.")

## 8. Your own file (optional)
Upload a CSV with a `text` column (optional `id`, `company`; other columns are kept). First run the dry run, then the paid cell with the same two switches.

In [ ]:
up2 = files.upload()                     # choose your CSV
INPUT_FILE = next(iter(up2))
!python triage.py "$INPUT_FILE" --dry-run

Then go back to cell 5 (keep `INPUT_FILE` as set above), set `CONFIRM_PAID_RUN = True`, run it, and run cells 6-7 again.

## Offline tests (optional, free)
162 checks with a fake embedding client: no key, no cost.

In [ ]:
!python test_triage_policy.py . 2>&1 | tail -3